# Active labeling with GeoCP-RS's $\hat q_j$ field

Empirical validation of one concrete downstream use of GeoCP-RS's per-pixel threshold field: **prioritize new labels at pixels with the highest $\hat q_j$**. Compared against the standard uncertainty-based acquisition (max prediction-set size) and random sampling.

## Protocol

For each HSI dataset $\in$ {Indian Pines, Pavia U., KSC}, 5 random seeds:
1. Train an \emph{initial} 3D-CNN on 100 stratified labeled pixels.
2. Split the remaining labeled pixels into `cal` (500), `test` (500), and `pool` (rest).
3. Run Standard-CP + GeoCP-RS (radius $r = 5$, bandwidth CV-selected) to obtain:
   - per-pixel prediction-set size $|\mathcal C(X_j)|$ at every pool point,
   - per-pixel threshold $\hat q_j$ at every pool point.
4. Three acquisition strategies select $K = 100$ pool pixels to label:
   - **Random** — uniform random $K$.
   - **Max-|C|** — top-$K$ by prediction-set size (classical uncertainty-based AL).
   - **Max-$\hat q_j$** — top-$K$ by GeoCP-RS's new output (our proposal).
5. Re-train a 3D-CNN on 100 + 100 = 200 pixels and evaluate test accuracy + IS.

Expected outcome: Max-$\hat q_j$ yields the largest test accuracy and IS improvement over the initial model, with reliability advantages in the metric (IS) that we actually care about.

**Runtime**: ~30--45 minutes on a Colab T4 (3 datasets × 5 seeds × 4 model trainings each = 60 trainings, ~30s each).

Outputs written to `/content/drive/MyDrive/active_labeling/`.

## 1. Setup + paths

In [ ]:
!pip install -q scikit-learn scipy matplotlib

import os, sys, time, gc, pickle, json, shutil
import numpy as np
import scipy.io as sio
import torch
print('python', sys.version.split()[0], '| torch', torch.__version__, '| cuda', torch.cuda.is_available())
if torch.cuda.is_available(): print('GPU:', torch.cuda.get_device_name(0))

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

DATA_DIR   = '/content/drive/MyDrive/sacp_geocp/datasets'   # reuse the same HSI .mat files
LOCAL_DIR  = '/content/active_labeling'
RESULTS    = f'{LOCAL_DIR}/results'
RUN_CACHE  = f'{LOCAL_DIR}/runs'
for d in [LOCAL_DIR, RESULTS, RUN_CACHE]: os.makedirs(d, exist_ok=True)
DRIVE_OUT  = '/content/drive/MyDrive/active_labeling'
try: os.makedirs(f'{DRIVE_OUT}/runs', exist_ok=True)
except OSError: pass

# Restore any cached results from Drive
def _restore():
    src = f'{DRIVE_OUT}/runs'
    if not os.path.isdir(src): return 0
    n = 0
    for f in os.listdir(src):
        if not f.endswith('.pkl'): continue
        dst = f'{RUN_CACHE}/{f}'
        if os.path.exists(dst): continue
        try: shutil.copy2(f'{src}/{f}', dst); n += 1
        except OSError as e: print(f'[restore] {f}: {e}')
    return n
print(f'[restore] {_restore()} cached per-run pkls')

python 3.12.13 | torch 2.10.0+cpu | cuda False
Mounted at /content/drive
[restore] 0 cached per-run pkls


## 2. Helpers (reused from hsi_alpha_sweep)

In [ ]:
import torch.nn as nn, torch.nn.functional as F, torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from scipy.spatial.distance import cdist
from scipy.signal import convolve2d

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class CNN3D(nn.Module):
    def __init__(self, input_channels, n_classes, patch_size=5):
        super().__init__()
        self.conv1 = nn.Conv3d(1, 20, (3,3,3), padding=0)
        self.pool1 = nn.Conv3d(20, 20, (3,1,1), stride=(2,1,1), padding=(1,0,0))
        self.conv2 = nn.Conv3d(20, 35, (3,3,3), padding=(1,0,0))
        self.pool2 = nn.Conv3d(35, 35, (3,1,1), stride=(2,1,1), padding=(1,0,0))
        self.conv3 = nn.Conv3d(35, 35, (3,1,1), padding=(1,0,0))
        self.conv4 = nn.Conv3d(35, 35, (2,1,1), stride=(2,1,1), padding=(1,0,0))
        self.patch_size = patch_size; self.input_channels = input_channels
        self.features_size = self._feat_size()
        self.fc = nn.Linear(self.features_size, n_classes)
    def _feat_size(self):
        with torch.no_grad():
            x = torch.zeros((1, 1, self.input_channels, self.patch_size, self.patch_size))
            x = self.pool1(self.conv1(x)); x = self.pool2(self.conv2(x))
            x = self.conv3(x); x = self.conv4(x)
            return int(np.prod(x.size()[1:]))
    def forward(self, x):
        x = F.relu(self.conv1(x)); x = self.pool1(x)
        x = F.relu(self.conv2(x)); x = self.pool2(x)
        x = F.relu(self.conv3(x)); x = F.relu(self.conv4(x))
        x = x.view(-1, self.features_size)
        return self.fc(x)

def aps_scores(probs, labels=None, rng=None):
    rng = rng or np.random
    n, K = probs.shape
    si = np.argsort(-probs, axis=1)
    sp = np.take_along_axis(probs, si, axis=1)
    cs = np.cumsum(sp, axis=1)
    U = rng.uniform(0, 1, size=(n, K))
    ss = cs - sp * U
    scores = np.zeros_like(ss)
    for i in range(n):
        scores[i, si[i]] = ss[i]
    if labels is not None:
        return np.array([scores[i, int(labels[i])] for i in range(n)])
    return scores

def conformal_quantile(scores, alpha):
    n = len(scores)
    return float(np.quantile(scores, np.ceil((n+1)*(1-alpha))/n))

def sacp_smooth_radius(score_map, H, W, valid_idx, radius, lmd=0.5, k_iter=1):
    N, K = score_map.shape
    if radius <= 0: return score_map.copy()
    mask = np.zeros(N, dtype=bool); mask[valid_idx] = True
    mask_2d = mask.reshape(H, W).astype(np.float64)
    score_2d = score_map.reshape(H, W, K).astype(np.float64)
    k_size = 2 * radius + 1
    kernel = np.ones((k_size, k_size), dtype=np.float64); kernel[radius, radius] = 0.0
    for _ in range(k_iter):
        den = convolve2d(mask_2d, kernel, mode='same', boundary='fill', fillvalue=0.0)
        smoothed = np.empty_like(score_2d)
        for cls in range(K):
            values = np.where(mask_2d > 0, score_2d[..., cls], 0.0)
            num = convolve2d(values, kernel, mode='same', boundary='fill', fillvalue=0.0)
            smoothed[..., cls] = np.where(den > 1e-10, num / np.maximum(den, 1e-10), 0.0)
        new_score = (1 - lmd) * score_2d + lmd * smoothed
        score_2d = np.where(mask_2d[..., None] > 0, new_score, score_2d)
    return score_2d.reshape(N, K)

def vwq(sorted_scores, d_mat, order, bw, alpha):
    log_w = -0.5 * (d_mat / bw) ** 2
    log_w -= log_w.max(axis=1, keepdims=True)
    w = np.exp(log_w); w_sorted = w[:, order]
    ws = w_sorted / w_sorted.sum(axis=1, keepdims=True)
    cum = np.cumsum(ws, axis=1)
    k_star = np.argmax(cum >= (1 - alpha), axis=1)
    return sorted_scores[k_star]

def extract_patches(hsi_chw, indices, patch_size=2):
    d, h, w = hsi_chw.shape
    padded = np.pad(hsi_chw, ((0,0),(patch_size,patch_size),(patch_size,patch_size)), mode='reflect')
    patches = np.zeros((len(indices), 1, d, 2*patch_size+1, 2*patch_size+1), dtype=np.float32)
    for e, idx in enumerate(indices):
        r, c = idx // w, idx % w
        patches[e, 0] = padded[:, r:r+2*patch_size+1, c:c+2*patch_size+1]
    return patches

DATASET_CONFIG = {
    'ip':  ('ip/Indian_pines_corrected.mat', 'indian_pines_corrected',
            'ip/Indian_pines_gt.mat', 'indian_pines_gt', 16, 200, 'Indian Pines'),
    'pu':  ('pu/PaviaU.mat', 'paviaU', 'pu/PaviaU_gt.mat', 'paviaU_gt', 9, 103, 'Pavia University'),
    'ksc': ('ksc/KSC.mat', 'KSC', 'ksc/KSC_gt.mat', 'KSC_gt', 13, 176, 'KSC'),
}
print('Helpers loaded.')

Helpers loaded.


## 3. Core AL experiment function

In [ ]:
# Config
N_INITIAL  = 100          # initial train set size
K_ACQUIRE  = 100          # how many pool pixels each strategy acquires
N_CAL      = 500
N_TEST     = 500
EPOCHS     = 100
ALPHA      = 0.10
LMD        = 0.5
R_FIXED    = 5            # paper shows r=5 is chosen by CV on most datasets; fixed here for speed
BW_GRID    = [3, 5, 7, 10, 15, 20, 30, 50, 100]

def train_cnn(X_tr, y_tr, bands, n_classes, epochs=EPOCHS):
    model = CNN3D(bands, n_classes, patch_size=5).to(device)
    loader = DataLoader(TensorDataset(torch.FloatTensor(X_tr), torch.LongTensor(y_tr)),
                        batch_size=64, shuffle=True)
    opt = optim.Adam(model.parameters(), lr=0.001); crit = nn.CrossEntropyLoss()
    for _ in range(epochs):
        model.train()
        for Xb, yb in loader:
            Xb, yb = Xb.to(device), yb.to(device)
            opt.zero_grad(); crit(model(Xb), yb).backward(); opt.step()
    return model

def get_probs(model, X):
    model.eval()
    out = []
    loader = DataLoader(TensorDataset(torch.FloatTensor(X)), batch_size=256, shuffle=False)
    with torch.no_grad():
        for (b,) in loader: out.append(torch.softmax(model(b.to(device)), dim=1).cpu().numpy())
    return np.concatenate(out)

def prediction_sets_and_is(probs, y_true, q_vec_or_scalar, alpha, rng):
    '''q_vec_or_scalar: either scalar (global threshold) or per-pixel vector.'''
    scores_all = aps_scores(probs, rng=rng)
    if np.isscalar(q_vec_or_scalar):
        q = np.full(len(y_true), q_vec_or_scalar)
    else:
        q = q_vec_or_scalar
    in_set = scores_all < q[:, None]
    sizes = in_set.sum(axis=1)
    covered = in_set[np.arange(len(y_true)), y_true.astype(int)]
    IS = float((sizes + (2.0/alpha) * (~covered).astype(np.float64)).mean())
    return float(sizes.mean()), float(covered.mean()), IS

def run_one(ds, seed):
    ckpt = f'{RUN_CACHE}/{ds}_seed{seed}.pkl'
    if os.path.exists(ckpt):
        with open(ckpt, 'rb') as f: return pickle.load(f)

    torch.manual_seed(seed*100+42); np.random.seed(seed*100+42)
    rng = np.random.RandomState(seed*100+42)

    hf, hk, gf, gk, n_classes, bands, nice = DATASET_CONFIG[ds]
    hsi = sio.loadmat(f'{DATA_DIR}/{hf}')[hk]
    gt  = sio.loadmat(f'{DATA_DIR}/{gf}')[gk]
    h, w, d = hsi.shape; N = h*w; K = n_classes
    hsi_norm = hsi.astype(np.float32)
    hsi_norm = (hsi_norm - hsi_norm.mean(axis=(0,1))) / (hsi_norm.max() + 1e-8)
    hsi_chw = hsi_norm.transpose(2, 0, 1)

    y_all = gt.reshape(N)
    labeled_idx = np.where(y_all > 0)[0]
    coords = np.column_stack([np.arange(N)//w, np.arange(N)%w]).astype(np.float32)

    rs = seed*100+42
    # Split: initial-train (100) + cal (500) + test (500) + pool (rest)
    y_lab = y_all[labeled_idx] - 1
    idx_tr, idx_rest = train_test_split(np.arange(len(labeled_idx)), train_size=N_INITIAL,
                                         stratify=y_lab, random_state=rs)
    idx_ca, idx_rest2 = train_test_split(idx_rest, train_size=N_CAL,
                                          stratify=y_lab[idx_rest], random_state=rs)
    idx_te, idx_pool = train_test_split(idx_rest2, train_size=N_TEST,
                                         stratify=y_lab[idx_rest2], random_state=rs)
    tr_gi = labeled_idx[idx_tr]; ca_gi = labeled_idx[idx_ca]
    te_gi = labeled_idx[idx_te]; pool_gi = labeled_idx[idx_pool]
    y_tr  = y_all[tr_gi]-1; y_ca = y_all[ca_gi]-1
    y_te  = y_all[te_gi]-1; y_pool = y_all[pool_gi]-1

    X_tr  = extract_patches(hsi_chw, tr_gi, 2)
    X_ca  = extract_patches(hsi_chw, ca_gi, 2)
    X_te  = extract_patches(hsi_chw, te_gi, 2)
    X_pool= extract_patches(hsi_chw, pool_gi, 2)

    # Step 1: train initial model
    t0 = time.time()
    model0 = train_cnn(X_tr, y_tr, bands, K)
    probs_ca0 = get_probs(model0, X_ca); probs_te0 = get_probs(model0, X_te)
    probs_pool0 = get_probs(model0, X_pool)
    acc0 = float(np.mean(np.argmax(probs_te0, axis=1) == y_te))

    # Step 2: compute Standard-CP quantile q_D and |C|_D at pool
    cal_true = aps_scores(probs_ca0, y_ca, rng=rng)
    q_D = conformal_quantile(cal_true, ALPHA)
    scores_pool = aps_scores(probs_pool0, rng=rng)
    sizes_D_pool = (scores_pool < q_D).sum(axis=1)

    # Step 3: SACP smoothed scores at r=R_FIXED, over cal ∪ test ∪ pool positions
    cal_all = aps_scores(probs_ca0, rng=rng)
    test_all = aps_scores(probs_te0, rng=rng)
    pool_all = aps_scores(probs_pool0, rng=rng)
    score_map = np.zeros((N, K))
    score_map[ca_gi] = cal_all; score_map[te_gi] = test_all; score_map[pool_gi] = pool_all
    valid_idx_all = np.concatenate([ca_gi, te_gi, pool_gi])
    fused = sacp_smooth_radius(score_map, h, w, valid_idx_all, radius=R_FIXED, lmd=LMD)
    fcu = np.array([fused[ca_gi[e], int(y_ca[e])] for e in range(len(idx_ca))])

    # Step 4: 5-fold CV (on cal) over bw to pick bw*, evaluated on cal itself; at alpha=0.10
    from sklearn.model_selection import KFold
    coords_ca = coords[ca_gi]
    kf = KFold(n_splits=5, shuffle=True, random_state=0)
    cv_is = {bw: [] for bw in BW_GRID}
    for f_tr, f_val in kf.split(np.arange(len(idx_ca))):
        tr_sc = fcu[f_tr]; val_sc = fused[ca_gi[f_val]]; val_y = y_ca[f_val]
        order = np.argsort(tr_sc); sorted_tr = tr_sc[order]
        d_val = cdist(coords_ca[f_val], coords_ca[f_tr])
        for bw in BW_GRID:
            q_val = vwq(sorted_tr, d_val, order, bw, ALPHA)
            in_set = val_sc < q_val[:, None]
            sizes = in_set.sum(axis=1); covered = in_set[np.arange(len(val_y)), val_y.astype(int)]
            cv_is[bw].append(float((sizes + (2.0/ALPHA)*(~covered)).mean()))
    best_bw = int(min(BW_GRID, key=lambda b: np.mean(cv_is[b])))

    # Step 5: compute q_hat_j at pool AND test positions
    order = np.argsort(fcu); sorted_cal = fcu[order]
    coords_pool = coords[pool_gi]; coords_te = coords[te_gi]
    def qhat_at(coords_target):
        BT = 500; n = len(coords_target); q = np.empty(n)
        for b0 in range(0, n, BT):
            b1 = min(b0+BT, n)
            dmat = cdist(coords_target[b0:b1], coords_ca)
            q[b0:b1] = vwq(sorted_cal, dmat, order, best_bw, ALPHA)
        return q
    qhat_pool = qhat_at(coords_pool)
    qhat_te0  = qhat_at(coords_te)

    # Initial model IS at test (GeoCP-RS)
    ftu0 = fused[te_gi]
    in_set0 = ftu0 < qhat_te0[:, None]
    sizes0 = in_set0.sum(axis=1); covered0 = in_set0[np.arange(len(y_te)), y_te.astype(int)]
    IS_init = float((sizes0 + (2.0/ALPHA)*(~covered0)).mean())

    # Step 6: three acquisition strategies, each picking K_ACQUIRE pool pixels
    results = {'ds': ds, 'seed': seed, 'nice': nice, 'best_bw': best_bw,
               'acc_initial': acc0, 'IS_initial': IS_init}
    pool_n = len(idx_pool)
    # Greedy 'diverse' selection: sort by q_hat descending, accept if ≥ bw* pixels
    # away from all already-selected points (farthest-point-style mask).
    def greedy_diverse_topk(scores, coords_pool_arr, k, min_dist):
        order = np.argsort(-scores)  # descending
        picked = []
        picked_coords = []
        for idx in order:
            if not picked:
                picked.append(idx); picked_coords.append(coords_pool_arr[idx]); continue
            d = np.min(np.linalg.norm(np.asarray(picked_coords) - coords_pool_arr[idx], axis=1))
            if d >= min_dist:
                picked.append(idx); picked_coords.append(coords_pool_arr[idx])
            if len(picked) >= k: break
        # Fallback: if diversity constraint left <k, fill with remaining top-q̂
        if len(picked) < k:
            rem = [i for i in order if i not in set(picked)][:k - len(picked)]
            picked.extend(rem)
        return np.array(picked[:k])

    strategies = {
        'random':          rng.choice(pool_n, K_ACQUIRE, replace=False),
        'max_cset':        np.argsort(sizes_D_pool)[-K_ACQUIRE:],
        'max_qhat':        np.argsort(qhat_pool)[-K_ACQUIRE:],
        'min_qhat':        np.argsort(qhat_pool)[:K_ACQUIRE],   # sanity-check: low uncertainty
        'max_qhat_div':    greedy_diverse_topk(qhat_pool, coords_pool, K_ACQUIRE, min_dist=best_bw),
    }
    for name, pick_idx in strategies.items():
        chosen_pool = pool_gi[pick_idx]
        new_tr_gi = np.concatenate([tr_gi, chosen_pool])
        new_y_tr  = y_all[new_tr_gi] - 1
        X_new_tr  = extract_patches(hsi_chw, new_tr_gi, 2)
        # Retrain
        model_s = train_cnn(X_new_tr, new_y_tr, bands, K)
        probs_te_s = get_probs(model_s, X_te)
        acc_s = float(np.mean(np.argmax(probs_te_s, axis=1) == y_te))
        # IS under GeoCP-RS at the SAME (r, bw) for fair comparison
        # Recompute on new model's cal+test+pool scores
        probs_ca_s = get_probs(model_s, X_ca)
        cal_all_s = aps_scores(probs_ca_s, rng=rng); test_all_s = aps_scores(probs_te_s, rng=rng)
        sm = np.zeros((N, K))
        sm[ca_gi] = cal_all_s; sm[te_gi] = test_all_s
        fused_s = sacp_smooth_radius(sm, h, w, np.concatenate([ca_gi, te_gi]), radius=R_FIXED, lmd=LMD)
        fcu_s = np.array([fused_s[ca_gi[e], int(y_ca[e])] for e in range(len(idx_ca))])
        order_s = np.argsort(fcu_s); sorted_s = fcu_s[order_s]
        q_te_s = np.empty(len(y_te))
        for b0 in range(0, len(y_te), 500):
            b1 = min(b0+500, len(y_te))
            q_te_s[b0:b1] = vwq(sorted_s, cdist(coords_te[b0:b1], coords_ca), order_s, best_bw, ALPHA)
        ftu_s = fused_s[te_gi]
        in_set = ftu_s < q_te_s[:, None]
        sizes = in_set.sum(axis=1); covered = in_set[np.arange(len(y_te)), y_te.astype(int)]
        IS_s = float((sizes + (2.0/ALPHA)*(~covered)).mean())
        results[f'acc_{name}']    = acc_s
        results[f'IS_{name}']     = IS_s
        results[f'gain_acc_{name}'] = acc_s - acc0
        results[f'gain_IS_{name}']  = (IS_init - IS_s) / IS_init * 100   # % reduction in IS (higher = better)
        del model_s, probs_te_s, probs_ca_s, X_new_tr
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()

    results['dt'] = time.time() - t0
    with open(ckpt, 'wb') as f: pickle.dump(results, f)
    try: shutil.copy2(ckpt, f'{DRIVE_OUT}/runs/{os.path.basename(ckpt)}')
    except OSError: pass
    return results

print('run_one defined')

run_one defined


## 4. Run across 3 datasets × 5 seeds (resumable)

In [5]:
DATASETS = ['ip', 'pu', 'ksc']
N_SEEDS  = 5

runs = []
t_all = time.time()
for ds in DATASETS:
    print(f'\n=== {DATASET_CONFIG[ds][6]} ({ds}) ===')
    for seed in range(N_SEEDS):
        r = run_one(ds, seed)
        runs.append(r)
        print(f'  seed={seed}  bw*={r["best_bw"]:>3}  acc0={r["acc_initial"]:.3f}')
        print(f'    Δacc :  rand={r["gain_acc_random"]:+.3f}  |C|={r["gain_acc_max_cset"]:+.3f}  '
              f'q̂={r["gain_acc_max_qhat"]:+.3f}  minq̂={r["gain_acc_min_qhat"]:+.3f}  q̂+div={r["gain_acc_max_qhat_div"]:+.3f}')
        print(f'    ΔIS% :  rand={r["gain_IS_random"]:+.2f}  |C|={r["gain_IS_max_cset"]:+.2f}  '
              f'q̂={r["gain_IS_max_qhat"]:+.2f}  minq̂={r["gain_IS_min_qhat"]:+.2f}  q̂+div={r["gain_IS_max_qhat_div"]:+.2f}  '
              f'({r["dt"]:.0f}s)')

print(f'\n==== DONE: {len(runs)} runs in {(time.time()-t_all)/60:.1f} min ====')


=== Indian Pines (ip) ===
  seed=0  bw*= 15  acc0=0.600
    Δacc :  rand=+0.130  |C|=+0.066  q̂=-0.040  minq̂=+0.002  q̂+div=+0.034
    ΔIS% :  rand=+18.82  |C|=+6.99  q̂=-9.96  minq̂=-8.57  q̂+div=+5.36  (654s)
  seed=1  bw*= 20  acc0=0.578
    Δacc :  rand=+0.022  |C|=+0.012  q̂=-0.008  minq̂=-0.036  q̂+div=+0.048
    ΔIS% :  rand=+10.37  |C|=+7.74  q̂=-2.53  minq̂=-23.84  q̂+div=-16.33  (610s)


KeyboardInterrupt: 

## 5. Aggregate + paired tests + plot

In [ ]:
import pandas as pd
from scipy import stats as sstats

rows = []
for r in runs:
    for strat in ('random', 'max_cset', 'max_qhat', 'min_qhat', 'max_qhat_div'):
        rows.append({'ds': r['ds'], 'seed': r['seed'], 'strategy': strat,
                     'acc_initial': r['acc_initial'], 'IS_initial': r['IS_initial'],
                     'acc_after':   r[f'acc_{strat}'], 'IS_after': r[f'IS_{strat}'],
                     'delta_acc':   r[f'gain_acc_{strat}'],
                     'delta_IS_pct': r[f'gain_IS_{strat}']})
df = pd.DataFrame(rows)
df.to_csv(f'{RESULTS}/al_per_run.csv', index=False)
print('Wrote al_per_run.csv')

# Pooled summary
summary = df.groupby('strategy').agg(
    delta_acc_mean=('delta_acc','mean'), delta_acc_std=('delta_acc','std'),
    delta_IS_mean=('delta_IS_pct','mean'), delta_IS_std=('delta_IS_pct','std'),
    n=('strategy','count')).reset_index()
print('\n=== Pooled (n per strategy) ===')
print(summary.round(3).to_string(index=False))

# Paired tests: q̂ vs random, q̂ vs |C|
print('\n=== Paired t-tests ===')
for metric in ('delta_acc', 'delta_IS_pct'):
    for strat_a, strat_b in [('max_qhat', 'random'), ('max_qhat', 'max_cset'),
                                ('max_qhat_div', 'max_qhat'), ('max_qhat_div', 'max_cset'),
                                ('min_qhat', 'random')]:
        a = df[df.strategy == strat_a].sort_values(['ds','seed'])[metric].values
        b = df[df.strategy == strat_b].sort_values(['ds','seed'])[metric].values
        t, p = sstats.ttest_rel(a, b)
        diff = (a - b).mean()
        print(f'  {metric:>15s}  {strat_a:>10s} vs {strat_b:>10s}:  mean diff = {diff:+.4f}  t = {t:+.2f}  p = {p:.4f}')

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 10,
                     'axes.spines.top': False, 'axes.spines.right': False})

strategies = ['min_qhat', 'random', 'max_cset', 'max_qhat', 'max_qhat_div']
labels = ['Min-$\\hat q_j$\n(sanity)', 'Random', 'Max-|C|\n(std. AL)',
          'Max-$\\hat q_j$\n(naive top-K)', 'Max-$\\hat q_j$\n+ diversity\n(ours)']
colors = ['#cc6677', '#9e9e9e', '#1f77b4', '#ffa0a0', '#d62728']

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))

# Panel 1: Δ accuracy
ax = axes[0]
for i, s in enumerate(strategies):
    vals = df[df.strategy == s].delta_acc.values
    ax.bar(i, vals.mean(), yerr=vals.std(ddof=1)/np.sqrt(len(vals)),
           color=colors[i], edgecolor='black', linewidth=0.5, capsize=4)
ax.set_xticks(range(len(strategies))); ax.set_xticklabels(labels, fontsize=9)
ax.set_ylabel('Δ test accuracy (after − before, ± s.e.m.)')
ax.set_title('Accuracy gain from +100 labels')
ax.axhline(0, color='k', lw=0.5); ax.grid(True, axis='y', alpha=0.3)

# Panel 2: IS improvement
ax = axes[1]
for i, s in enumerate(strategies):
    vals = df[df.strategy == s].delta_IS_pct.values
    ax.bar(i, vals.mean(), yerr=vals.std(ddof=1)/np.sqrt(len(vals)),
           color=colors[i], edgecolor='black', linewidth=0.5, capsize=4)
ax.set_xticks(range(len(strategies))); ax.set_xticklabels(labels, fontsize=9)
ax.set_ylabel('IS reduction (%, higher = better, ± s.e.m.)')
ax.set_title('IS gain under GeoCP-RS from +100 labels')
ax.axhline(0, color='k', lw=0.5); ax.grid(True, axis='y', alpha=0.3)

fig.suptitle(f'Active labeling: 3 HSI datasets × {N_SEEDS} seeds (n = {len(df)//len(strategies)} per bar)', y=1.02)
fig.tight_layout()
out_fig = f'{RESULTS}/fig_active_learning.png'
fig.savefig(out_fig, dpi=180, bbox_inches='tight')
plt.show()
print(f'Saved {out_fig}')

## 6. Push outputs to Drive

In [ ]:
for f in os.listdir(RESULTS):
    try: shutil.copy2(f'{RESULTS}/{f}', f'{DRIVE_OUT}/{f}')
    except OSError as e: print(f'Drive copy failed for {f}: {e}')
print('Outputs in', DRIVE_OUT)